Select the best model based on precision, recall, and F1-score, applying custom weighting strategies for imbalanced datasets and justifying selection using a detailed performance analysis.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42
    ),
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(probability=True, class_weight="balanced", random_state=42)
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)

print("All models trained successfully.")

All models trained successfully.


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [2]:
df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

print("Dataset shape:", df.shape)
print("Class distribution:")
print(pd.Series(y).value_counts().sort_index())

Dataset shape: (79, 6)
Class distribution:
0    18
1    61
Name: count, dtype: int64


In [3]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

Training samples: 47
Validation samples: 16
Testing samples: 16


In [4]:
def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

In [7]:
precision_weight = 0.20
recall_weight = 0.40
f1_weight = 0.40

results = []

for name, model in models.items():
    pred = model.predict(X_val)

    precision, recall, f1 = calculate_metrics(y_val, pred)

    custom_score = (
        precision_weight * precision
        + recall_weight * recall
        + f1_weight * f1
    )

    results.append({
        "Model": name,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Custom Score": custom_score
    })

results_df = pd.DataFrame(results).sort_values(
    "Custom Score", ascending=False
)

print(results_df.round(4).to_string(index=False))

              Model  Precision  Recall  F1 Score  Custom Score
Logistic Regression     0.9167  0.9167    0.9167        0.9167
                KNN     0.7500  1.0000    0.8571        0.8929
                SVM     0.9091  0.8333    0.8696        0.8630
      Random Forest     0.8333  0.8333    0.8333        0.8333
      Decision Tree     0.8333  0.8333    0.8333        0.8333


In [8]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Selected model:", best_model_name)
print("Precision:", round(results_df.iloc[0]["Precision"], 4))
print("Recall:", round(results_df.iloc[0]["Recall"], 4))
print("F1 Score:", round(results_df.iloc[0]["F1 Score"], 4))
print("Custom Score:", round(results_df.iloc[0]["Custom Score"], 4))

Selected model: Logistic Regression
Precision: 0.9167
Recall: 0.9167
F1 Score: 0.9167
Custom Score: 0.9167


In [9]:
test_pred = best_model.predict(X_test)

test_precision, test_recall, test_f1 = calculate_metrics(
    y_test, test_pred
)

tn, fp, fn, tp = confusion_matrix(
    y_test, test_pred, labels=[0, 1]
).ravel()

print("Final Test Results")
print("Selected model:", best_model_name)
print("Precision:", round(test_precision, 4))
print("Recall:", round(test_recall, 4))
print("F1 Score:", round(test_f1, 4))
print("True Positives:", tp)
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)

Final Test Results
Selected model: Logistic Regression
Precision: 0.7692
Recall: 0.8333
F1 Score: 0.8
True Positives: 10
True Negatives: 1
False Positives: 3
False Negatives: 2


In [10]:
print("Performance Analysis")

if test_recall >= 0.90:
    print("Recall is high: the model detects most positive cases in this test set.")
else:
    print("Recall could be improved: some positive cases may be missed.")

if test_precision >= 0.90:
    print("Precision is high: most predicted positive cases are actually positive.")
else:
    print("Precision could be improved: some predicted positive cases are false alarms.")

if test_f1 >= 0.80:
    print("F1-score indicates a good balance between precision and recall.")
else:
    print("F1-score suggests that precision and recall need further improvement.")

print("\nValidation ranking:")
print(results_df[["Model", "Precision", "Recall", "F1 Score", "Custom Score"]]
      .round(4).to_string(index=False))

Performance Analysis
Recall could be improved: some positive cases may be missed.
Precision could be improved: some predicted positive cases are false alarms.
F1-score indicates a good balance between precision and recall.

Validation ranking:
              Model  Precision  Recall  F1 Score  Custom Score
Logistic Regression     0.9167  0.9167    0.9167        0.9167
                KNN     0.7500  1.0000    0.8571        0.8929
                SVM     0.9091  0.8333    0.8696        0.8630
      Random Forest     0.8333  0.8333    0.8333        0.8333
      Decision Tree     0.8333  0.8333    0.8333        0.8333
